<a href="https://colab.research.google.com/github/tarannump096-cpu/NLP/blob/main/Malware_Dataset_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
pip install pandas numpy scikit-learn nltk

In [8]:
import pandas as pd
import numpy as np
import re
import nltk
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

nltk.download('stopwords')
from nltk.corpus import stopwords

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [9]:
df = pd.read_csv("/content/Malware dataset.csv")
print(df.head())

                                                hash  millisecond  \
0  42fb5e2ec009a05ff5143227297074f1e9c6c3ebb9c914...            0   
1  42fb5e2ec009a05ff5143227297074f1e9c6c3ebb9c914...            1   
2  42fb5e2ec009a05ff5143227297074f1e9c6c3ebb9c914...            2   
3  42fb5e2ec009a05ff5143227297074f1e9c6c3ebb9c914...            3   
4  42fb5e2ec009a05ff5143227297074f1e9c6c3ebb9c914...            4   

  classification  state  usage_counter        prio  static_prio  normal_prio  \
0        malware      0              0  3069378560        14274            0   
1        malware      0              0  3069378560        14274            0   
2        malware      0              0  3069378560        14274            0   
3        malware      0              0  3069378560        14274            0   
4        malware      0              0  3069378560        14274            0   

   policy  vm_pgoff  ...  nivcsw  min_flt  maj_flt  fs_excl_counter  \
0       0         0  ...       0 

In [10]:
print(df.columns)

Index(['hash', 'millisecond', 'classification', 'state', 'usage_counter',
       'prio', 'static_prio', 'normal_prio', 'policy', 'vm_pgoff',
       'vm_truncate_count', 'task_size', 'cached_hole_size', 'free_area_cache',
       'mm_users', 'map_count', 'hiwater_rss', 'total_vm', 'shared_vm',
       'exec_vm', 'reserved_vm', 'nr_ptes', 'end_data', 'last_interval',
       'nvcsw', 'nivcsw', 'min_flt', 'maj_flt', 'fs_excl_counter', 'lock',
       'utime', 'stime', 'gtime', 'cgtime', 'signal_nvcsw'],
      dtype='object')


In [15]:
stop_words = set(stopwords.words('english'))

def clean_text(input_text):
    input_text = input_text.lower()
    input_text = re.sub(r'http\S+', '', input_text)  # remove URLs
    input_text = re.sub(r'[^a-zA-Z]', ' ', input_text)  # remove symbols
    words = input_text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

df['clean_hash'] = df['hash'].apply(clean_text)

In [18]:
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_hash'], df['hash'], test_size=0.2, random_state=42
)

In [19]:
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1,2))),
    ('model', LogisticRegression())
])

pipeline.fit(X_train, y_train)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
                ('model', LogisticRegression())])

In [20]:
y_pred = pipeline.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 1.0
                                                                  precision    recall  f1-score   support

024b27972a6b3a1535510e9c0f154fb1a8e3a2afb25d5c30d2f6a9d23424d925       1.00      1.00      1.00       199
025c63d266e05d9e3bd57dd9ebd0abe904616f569fe4e2b78cf2ac52493cb460       1.00      1.00      1.00       179
0602834d897fe3f3314586ae867aed63f3757be01b7f0354c8626519d8575453       1.00      1.00      1.00       201
079277b8b6049c06806b79216901d0e9ff473bfe2c2454aa8a496515167eca40       1.00      1.00      1.00       180
1117d14765e9169184cc931f7a417a460898e4b0d8f3c86562065fc82f5866ce       1.00      1.00      1.00       199
1119f652a5b1f04e98835e2b2ed56efe990e9073aa7dcb39251c98e21d335abd       1.00      1.00      1.00       190
116ae92ecfacb70146fe643d92878e522f71af393702f3b66d2135a06bcff57f       1.00      1.00      1.00       198
12fbe832590c8d44b1687b178450d49190c1a9d8e61c80a090f461168dd0bf8f       1.00      1.00      1.00       239
156a617d84b92c1611e153ebaa1fc2e

In [21]:
sample = ["powershell -nop -exec bypass download malicious script"]
print("Prediction:", pipeline.predict(sample))

Prediction: ['12fbe832590c8d44b1687b178450d49190c1a9d8e61c80a090f461168dd0bf8f']
